# 09 - قراءة وتحليل بيانات JSON شبه المهيكلة (JSON Data Ingestion & JQ Schema)

مرحباً بك في الدرس التاسع من مسار **Data Ingestion** ضمن كورس **RAG Mastery**!

في هذا الدرس، نتعلم كيفية استيراد وتحليل بيانات **JSON** شبه المهيكلة والمتداخلة:
1. **`JSONLoader` مع `jq_schema`**: استخراج الكيانات المتداخلة ككائنات `Document` مباشرة باستخدام لغة استعلامات `jq`.
2. **المعالجة الدلالية الذكية المخصصة عبر بايثون**: تسطيح الـ JSON إلى جمل لغوية طبيعية يفهمها نموذج التضمين بدقة، مع إثراء الـ Metadata للفلترة.
3. **حل تطبيق المحاضرة**: معالجة سجلات الأحداث وسجلات الويب (`events.json`) واستخراج الطابع الزمني وهوية المستخدم.

## 1. استيراد المكتبات والتحقق من البيئة

In [ ]:
import json
from pathlib import Path
from typing import List, Dict, Any

import jq
from langchain_core.documents import Document
from langchain_community.document_loaders import JSONLoader

print("JSONLoader & JQ imported successfully!")

## 2. فحص ملفات البيانات (`company_data.json` و `events.json`)

لدينا ملفان في مجلد `data/`:
- `company_data.json`: يمثل هيكل شركة متداخلاً (أقسام، موظفون، مهارات، مشاريع).
- `events.json`: يمثل سجل أحداث تصفح ومشاريع مستخدمين على النظام.

In [ ]:
from pathlib import Path
data_dir = Path('data') if Path('data').exists() else Path('../../data')
company_file = (data_dir / 'json' / 'company_data.json').resolve()
events_file = (data_dir / 'json' / 'events.json').resolve()

print(f"Company JSON exists: {company_file.exists()} (Size: {company_file.stat().st_size} bytes)")
print(f"Events JSON exists: {events_file.exists()} (Size: {events_file.stat().st_size} bytes)")

# استعراض محتوى ملف الشركة
with open(company_file, 'r', encoding='utf-8') as f:
    sample_data = json.load(f)
print('\n--- Company Data Preview ---')
print(json.dumps(sample_data, indent=2))


## 3. الطريقة الأولى: استخراج الكائنات عبر `JSONLoader` و `jq_schema` (Method 1)

نستخدم مسار `jq_schema=".employees[]"` لاستهداف مصفوفة الموظفين واستخراج كل موظف كـ `Document` مستقل.

In [ ]:
try:
    loader = JSONLoader(
        file_path=str(company_file),
        jq_schema=".employees[]",
        text_content=False
    )
    employee_docs = loader.load()

    print(f"[+] Successfully extracted {len(employee_docs)} employee documents.\n")
    for i, doc in enumerate(employee_docs, 1):
        print(f"--- Document #{i} [seq_num: {doc.metadata.get('seq_num')}] ---")
        print(f"Page Content:\n{doc.page_content}")
        print(f"Metadata: {doc.metadata}\n")
except Exception as e:
    print(f"[-] Error loading JSON with jq: {e}")

## 4. الطريقة الثانية: المعالجة الدلالية الذكية المخصصة عبر بايثون (Semantic Entity Extraction)

بدلاً من الاحتفاظ بأقواس الـ JSON، نقوم بصياغة سرد لغوي طبيعي مترابط للـ LLM، ونستخرج المعرفات كـ Metadata قابلة للفلترة.

In [ ]:
def parse_company_json_intelligently(file_path: str) -> List[Document]:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    company = data.get("company_name", "Unknown")
    departments = ", ".join(data.get("departments", []))
    documents: List[Document] = []

    for emp in data.get("employees", []):
        emp_id = emp.get("id")
        name = emp.get("name")
        role = emp.get("role")
        skills = ", ".join(emp.get("skills", []))
        
        projects_list = [
            f"{p['name']} (Status: {p['status']})"
            for p in emp.get("projects", [])
        ]
        projects_text = "; ".join(projects_list)

        # بناء سياق سردي واضح لنماذج التضمين
        content = (
            f"Employee Profile: {name} (ID: {emp_id})\n"
            f"Company: {company} | Departments: {departments}\n"
            f"Professional Role: {role}\n"
            f"Technical Skills: {skills}\n"
            f"Assigned Projects: {projects_text}"
        )

        metadata = {
            "source": Path(file_path).name,
            "employee_id": emp_id,
            "name": name,
            "role": role,
            "company": company,
            "skills_count": len(emp.get("skills", [])),
            "projects_count": len(emp.get("projects", []))
        }

        documents.append(Document(page_content=content, metadata=metadata))

    return documents

custom_employee_docs = parse_company_json_intelligently(str(company_file))
print(f"[+] Generated {len(custom_employee_docs)} custom semantic documents.\n")
for doc in custom_employee_docs:
    print(f"--- {doc.metadata['name']} [{doc.metadata['role']}] ---")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}\n")

## 5. تطبيق عملي: معالجة سجلات الأحداث (`events.json`)

تنفيذ تمرين المحاضرة لتحليل سجلات الـ Audit Logs واستخراج الطابع الزمني (`timestamp`)، ونوع الحدث (`event`)، ومعرف المستخدم (`user_id`).

In [ ]:
def parse_event_logs(file_path: str) -> List[Document]:
    with open(file_path, "r", encoding="utf-8") as f:
        events = json.load(f)

    event_docs: List[Document] = []
    for idx, ev in enumerate(events, 1):
        # بناء سياق وصفي لكل حدث
        details = [f"{k}: {v}" for k, v in ev.items() if k not in ["event", "timestamp", "user_id"]]
        details_str = " | ".join(details)

        content = (
            f"System Event: {ev.get('event')}\n"
            f"Initiated By User: {ev.get('user_id')}\n"
            f"Timestamp: {ev.get('timestamp')}\n"
            f"Event Details: {details_str}"
        )

        metadata = {
            "source": Path(file_path).name,
            "event_type": ev.get("event"),
            "user_id": ev.get("user_id"),
            "timestamp": ev.get("timestamp"),
            "page": ev.get("page", ""),
            "status": ev.get("status", "")
        }

        event_docs.append(Document(page_content=content, metadata=metadata))

    return event_docs

event_documents = parse_event_logs(str(events_file))
print(f"[+] Parsed {len(event_documents)} event logs into Documents.\n")
for doc in event_documents:
    print(f"[{doc.metadata['timestamp']}] {doc.metadata['event_type']} by {doc.metadata['user_id']}")
    print(f"Metadata: {doc.metadata}")
    print(f"Content:\n{doc.page_content}\n")

## 6. الخلاصة والمقارنة السريعة

| الأسلوب | الاستخدام الأفضل | المزايا | الملاحظات |
| :--- | :--- | :--- | :--- |
| **`JSONLoader (jq_schema)`** | استخراج سريع لكائنات مصفوفات JSON | سطر واحد ومرونة jq العالية | يبقى النص بصيغة JSON خام |
| **`Custom Semantic Parser`** | أنظمة RAG المؤسسية المتقدمة | سياق طبيعي مترابط للـ Embedding وفلترة كاملة | يتطلب كتابة كود مخصص لكل مخطط JSON |